In [1]:
import numpy as np
import pandas as pd
import rasterio
print("ok")



ok


In [2]:
pts = pd.read_csv("data/occurrences_thinned.csv")
pres = pts[["decimalLongitude", "decimalLatitude"]].copy()
pres.columns = ["lon", "lat"]
pres["presence"] = 1

layers = {
    "bio_1":  "data/bio/wc2.1_10m_bio_1.tif",
    "bio_4":  "data/bio/wc2.1_10m_bio_4.tif",
    "bio_12": "data/bio/wc2.1_10m_bio_12.tif",
    "bio_15": "data/bio/wc2.1_10m_bio_15.tif",
    "elev":   "data/elev/wc2.1_10m_elev.tif",
}
print(len(pres), "presence points")

102 presence points


In [5]:
def add_values(df):
    for name, path in layers.items():
        with rasterio.open(path) as src:
            arr = src.read(1).astype("float64")
            t = src.transform
            nodata = src.nodata
        cols = np.floor((df["lon"].values - t.c) / t.a).astype(int)
        rows = np.floor((df["lat"].values - t.f) / t.e).astype(int)
        ok = (rows >= 0) & (rows < arr.shape[0]) & (cols >= 0) & (cols < arr.shape[1])
        vals = np.full(len(df), np.nan)
        vals[ok] = arr[rows[ok], cols[ok]]
        if nodata is not None:
            vals[vals == nodata] = np.nan
        vals[vals < -1e30] = np.nan
        df[name] = vals
    return df

This function looks up the value of each map layer at each point. It reads the layer as a grid of numbers, converts each longitude and latitude into a row and column, and takes the number at that spot. Missing values are marked as nan.

In [6]:
test = add_values(pres.head(5).copy())
print(test)

         lon        lat  presence     bio_1       bio_4  bio_12     bio_15  \
0  76.998859  43.103908         1  0.300604  945.116028   469.0  53.215511   
1  77.948354  32.328899         1 -4.281969  881.312927   393.0  57.594460   
2  78.452742  42.517869         1  6.298958  956.603943   498.0  43.275745   
3  77.923801  34.032465         1 -7.416490  962.465332   115.0  84.075508   
4  78.259771  42.110389         1 -8.299177  931.400635   346.0  73.726852   

     elev  
0  2426.0  
1  4864.0  
2  1679.0  
3  5227.0  
4  3929.0  


The results of above cells show the presence of 5 snow leopards and their longitude, latitude and 5 map values (temperature, seasonality, rainfall, rain seasonality, elevation)

In [7]:
rng = np.random.default_rng(42)
n = 20000
bg = pd.DataFrame({
    "lon": rng.uniform(65, 110, n),
    "lat": rng.uniform(25, 50, n),
})
bg["presence"] = 0
bg = add_values(bg).dropna()
bg = bg.sample(5000, random_state=42)
print(len(bg), "background points")

5000 background points


In the above cell we have taken 20000 random background points and then remove the ones which are not useful and then we have taken 5000 points from the remaining points.

In [8]:
pres = add_values(pres)
print(pres.isna().sum())
pres = pres.dropna()

table = pd.concat([pres, bg], ignore_index=True)
table.to_csv("data/model_table.csv", index=False)
print(table.groupby("presence").size())
print(table.groupby("presence")[["bio_1", "bio_12", "elev"]].mean().round(1))

lon         0
lat         0
presence    0
bio_1       0
bio_4       0
bio_12      0
bio_15      0
elev        0
dtype: int64
presence
0    5000
1     102
dtype: int64
          bio_1  bio_12    elev
presence                       
0           6.9   429.0  1952.2
1          -1.5   369.1  3610.6


In the above cell we have added both tables of 102 snow leopard points and 5000 background points.The presence points are about 8 °C colder and about 1,650 m higher then background points which indicates that snow leopard may exist there.
